# Đo hybrid BM25 + dense — nâng trần pool hay không?

EDA §15.7: reranker đã **bão hoà pool** (pool 20, k=10 ăn 100% khoảng trống). Năng lực
xếp hạng hết là chỗ nghẽn. Đường duy nhất còn lại của dense là kéo gold từ hạng 51+ vào
top-50 — nâng **trần**.

Notebook này KHÔNG chạy cross-encoder: chỉ đo trần, nhanh hơn ~50 lần. Reranker lo phần
sau với hiệu suất đã biết (ăn ~72% khoảng trống → +1 điểm trần ≈ +0,7 điểm R@5).

**Ngưỡng chốt trước khi chạy** — mốc phải vượt là BM25 pool 100 (trần 97,3%, giá 0 params):

| trần hybrid @50 | kết luận |
|---|---|
| < 96,2% | dense kéo tụt → vứt |
| 96,2–97,3% | không hơn pool 100 vốn miễn phí params → vứt |
| ≥ 98% | +1,3 điểm R@5, đáng 135M params → gắn |

Cần gắn: `official` · `legalqa-v10-src` · `legalqa-warmup` · GPU T4 x2 · Internet ON

## 1 — Nạp codebase từ Dataset

Codebase vào bằng **Kaggle Dataset** `legalqa-v10-src`, không clone GitHub: repo private,
fine-grained PAT qua Kaggle Secrets tốn hai vòng debug mà vẫn 404.

Cần gắn: **Input → Add Input → Datasets → `legalqa-v10-src`**

Dòng `codebase:` in ra commit của bản đang chạy. Đối chiếu với `git log -1` trên máy dev —
lệch nghĩa là dataset cũ, chạy `./scripts/push_dataset.sh` rồi *Add Input* lại bản mới.


In [ ]:
import os, sys, shutil, zipfile, subprocess
from pathlib import Path

# Không clone GitHub nữa: repo private, fine-grained PAT + Kaggle Secrets tốn hai vòng
# debug mà vẫn 404. Codebase đi vào bằng Kaggle Dataset — không auth, chạy được cả khi
# Internet OFF. Đẩy bản mới bằng scripts/push_dataset.sh trên máy dev.
SRC = None
for cand in Path("/kaggle/input").glob("**/*"):
    if cand.name == "legalqa" and (cand / "bm25.py").exists():
        SRC = cand.parent.parent          # .../src/legalqa -> gốc repo
        break
if SRC is None:
    # Kaggle đôi khi giữ nguyên .zip thay vì bung; tự bung vào working.
    z = next(Path("/kaggle/input").glob("**/src.zip"), None)
    assert z, "Không thấy codebase trong /kaggle/input — chưa gắn dataset legalqa-v10-src"
    with zipfile.ZipFile(z) as f:
        f.extractall("/kaggle/working/unz")
    SRC = Path("/kaggle/working/unz")

# Copy sang working: scripts ghi bm25.pkl/chunks.jsonl, /kaggle/input là read-only.
REPO = Path("/kaggle/working/repo")
shutil.rmtree(REPO, ignore_errors=True)
shutil.copytree(SRC, REPO)

!pip -q install transformers==4.44.2 sentencepiece

os.chdir(REPO)
sys.path.insert(0, "src")

import torch
print("codebase:", (REPO / "COMMIT.txt").read_text().strip() if (REPO / "COMMIT.txt").exists() else "KHÔNG có COMMIT.txt")
print("GPU     :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ — bật Accelerator")
print("số GPU  :", torch.cuda.device_count())

# P100 (sm_60) không chạy được fp16 với torch bản trên Kaggle. Code tự tụt về
# fp32 nên không vỡ, nhưng chậm ~2x — báo to để biết mà đổi Accelerator.
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability()
    print("sm      :", f"{cap[0]}.{cap[1]}", "(fp16 OK)" if cap[0] >= 7 else "(fp16 KHÔNG chạy -> fp32, đổi Accelerator sang T4 x2)")


## 2 — Tìm dữ liệu thi

Dò thay vì bắt gõ tay đường dẫn: tên dataset khác nhau tuỳ bạn gắn competition hay upload zip.
Mốc nhận dạng là thư mục `selected-contexts/` — thứ duy nhất chắc chắn có.

In [ ]:
from pathlib import Path

cands = [p.parent for p in Path("/kaggle/input").glob("**/selected-contexts") if p.is_dir()]
assert cands, "Không thấy selected-contexts/ trong /kaggle/input — chưa gắn dữ liệu thi"
DATA_DIR = cands[0]

train = DATA_DIR / "train.json"
assert train.exists(), f"Thiếu train.json cạnh selected-contexts/ trong {DATA_DIR}"

os.environ["DATA_DIR"] = str(DATA_DIR)
os.environ["TRAIN_JSON"] = str(train)
os.environ["CLEAN_DIR"] = "/kaggle/working/clean"
os.environ["PYTHONPATH"] = "src"

n = len(list((DATA_DIR / "selected-contexts").glob("*.json")))
print(f"DATA_DIR  {DATA_DIR}")
print(f"corpus    {n:,} file  (máy dev: 8.532)")

## 3 — clean → chunk → gold (CPU, ~12 phút)

Ba số phải khớp máy dev, lệch là dữ liệu khác bản — mọi số sau đó vô nghĩa:
`Document 8.512` · `Chunk 287.358` · `gold 6.250/7.000 = 89,3%`

In [ ]:
!mkdir -p $CLEAN_DIR
!python scripts/clean_data.py
!python scripts/build_chunks.py
!python scripts/build_gold.py
# warmup.json: 500 câu CÓ Reference answer và KHÔNG nằm trong train. Mọi ngưỡng pool/K
# đang chốt trên train -> cần tập độc lập kiểm lại, nếu không là chốt tham số trên chính
# tập đã dùng để quan sát. Máy dev dựng được 456/500 = 91,2%.
#
# Nó KHÔNG nằm cạnh train.json (dataset `official` chỉ có official/) -> dò cả /kaggle/input.
from pathlib import Path
warm = next(Path("/kaggle/input").glob("**/warmup.json"), None)
if warm:
    !TRAIN_JSON=$warm GOLD_OUT=gold_warmup.jsonl python scripts/build_gold.py
    os.environ["CO_WARMUP"] = "1"
else:
    os.environ["CO_WARMUP"] = ""
    print("KHÔNG thấy warmup.json — gắn dataset legalqa-warmup để có tập độc lập")


## 4 — Bi-encoder: e5-small, và chỉ nó

Lượt đo trước đã loại xong ba ứng viên, không đo lại:

| model | params | kết quả / lý do loại |
|---|---:|---|
| `intfloat/multilingual-e5-small` | 118M | **trần hybrid 97,9%** — thắng |
| `bkai/vietnamese-bi-encoder` | 135M | trần 97,6%; dense đơn 83,1%@10 *kém cả BM25*; encode chậm 3,4× |
| `dangvantuan/vietnamese-embedding`, `keepitreal/vietnamese-sbert` | 135M | **cùng một checkpoint PhoBERT** với bkai — params trùng đến từng đơn vị (134.998.272) |
| `AITeamVN/Vietnamese_Embedding`, `mainguyen9/vietlegal-e5` | 568M, 560M | cộng reranker 568M + Qwen 3,09B = 4,23B, **vượt trần 226M** |

Vì sao cả họ PhoBERT bị loại vì kiến trúc chứ không vì chất lượng: `max_position_embeddings
= 258`. Chunk 256 **từ** tiếng Việt ≈ 380-520 subword, nên PhoBERT không bao giờ đọc nổi
một chunk trọn vẹn. Trên benchmark Zalo (passage ngắn) nó chỉ kém 1,6 điểm A@1; trên chunk
của ta nó kém 9,1 điểm @10. Cùng model, hai kết luận trái nhau, chỉ vì độ dài đầu vào.


In [ ]:
import os, json
from huggingface_hub import snapshot_download
from transformers import AutoConfig

M = "intfloat/multilingual-e5-small"
P = snapshot_download(M)
cfg = AutoConfig.from_pretrained(P)
h, L, V = cfg.hidden_size, cfg.num_hidden_layers, cfg.vocab_size
print(f"{M}  h={h} L={L} vocab={V:,}  ~{(12*L*h*h + V*h)/1e6:.0f}M")
# max_position_embeddings là thứ chặn cứng max_len, xem trước khi xin 512.
print("max_position_embeddings:", cfg.max_position_embeddings)
os.environ["E5"] = P


## 5 — Đo lại ở max_len 256: dựng lại mốc

`BiEncoder(max_len=256)` là mặc định của lượt trước, và nó **cắt mất ~40% mỗi chunk**:
256 từ tiếng Việt ≈ 380-520 subword. Con số 97,9% đã đo là đo trên chunk bị cắt.

Cell này chạy lại đúng cấu hình cũ để hai lượt nằm trong **cùng một lần chạy** — nếu chỉ
so với số cũ qua log thì mọi thay đổi codebase ở giữa đều trộn vào chênh lệch.

Phải ra lại đúng: `BM25 96,7@50` · `hybrid 97,9@50`. Lệch là có gì khác đã đổi.


In [ ]:
!python scripts/eval_hybrid.py --dense-model $E5 --n-cau 1000 --deep 1000 --batch-size 256 --max-len 256

## 6 — max_len 512: đọc trọn chunk

Đổi một tham số duy nhất. Giá: encode gấp ~2 lần (455s → ~15 phút). Cache `.npy` có
`_L512` trong tên nên không nạp nhầm ma trận 256 của cell 5.

Ngưỡng chốt trước:

| chênh trần @50 so với cell 5 | quyết |
|---|---|
| ≥ +0,5 điểm | lấy 512, 8 phút encode thêm là rẻ |
| −0,5 … +0,5 | giữ 256 — đuôi chunk không mang thông tin định vị, tiết kiệm một nửa thời gian |
| ≤ −0,5 | bất thường, phải tìm hiểu chứ không chốt: đọc nhiều hơn mà tệ hơn nghĩa là mean pooling bị loãng bởi đuôi chunk |


In [ ]:
!python scripts/eval_hybrid.py --dense-model $E5 --n-cau 1000 --deep 1000 --batch-size 256 --max-len 512

## 7 — Nếu 512 thắng: kiểm lại bằng cross-encoder thật

Ước "+1 điểm trần ≈ +0,7 điểm R@5" là suy từ §15.6, **không phải đo**. Cell này đo thẳng,
nhưng chỉ đáng chạy sau khi đã chốt max_len — chạy trước là chốt trên cấu hình sắp bỏ.


In [ ]:
# Cần wiring hybrid vào eval_rerank.py trước (hiện nó chỉ nhận pool BM25).
# !python scripts/eval_rerank.py --model $MODEL --pool 50 --n-cau 500 --batch-size 32